# Sanity Check for DRR Renderers
This notebook loads a single CT volume and renders it across all 5 engines (Plastimatch, Monte Carlo, DVR, DiffDRR, DeepDRR) for visual inspection.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import matplotlib.pyplot as plt
from pathlib import Path

from utils.logger import setup_logger, get_logger
from data import load_ct_volume, make_diffdrr_subject
from renderers.config import DEFAULT_GEOMETRY
from renderers import (
    build_dvr_renderer,
    build_diffdrr_renderer,
    generate_plastimatch_drr,
    generate_mc_drr,
    generate_deepdrr_drr,
)

setup_logger()
logger = get_logger("sanity_check")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Load a single CT volume
target_size = 128
res = 200 # DRR resolution

logger.info("Loading CT volume...")
volume_tensor, subject, voxel_spacing = load_ct_volume(target_size=target_size)

if subject is None:
    subject = make_diffdrr_subject(volume_tensor, voxel_spacing)

vol = volume_tensor.to(device)
images = {}

In [ ]:
# 2. Define views to test
from renderers.config import GeometryConfig

views = {
    "AP": GeometryConfig(
        view_dir_x=0.0, view_dir_y=1.0, view_dir_z=0.0,
        isocenter_x=0.0, isocenter_y=0.0, isocenter_z=0.0
    ),
    "Lateral": GeometryConfig(
        view_dir_x=1.0, view_dir_y=0.0, view_dir_z=0.0,
        isocenter_x=0.0, isocenter_y=0.0, isocenter_z=0.0
    ),
    "Oblique (Offset Isocenter)": GeometryConfig(
        view_dir_x=0.707, view_dir_y=0.707, view_dir_z=0.0,
        isocenter_x=10.0, isocenter_y=-10.0, isocenter_z=5.0
    )
}

results = {}

for view_name, geo in views.items():
    logger.info(f"\n{'='*40}\nRendering View: {view_name}\n{'='*40}")
    images = {}
    
    logger.info("1. Rendering with Plastimatch (Ground Truth)...")
    img_plastimatch = generate_plastimatch_drr(volume_tensor, voxel_spacing, res, device, geometry=geo)
    if img_plastimatch is not None:
        images["Plastimatch"] = img_plastimatch.cpu().squeeze().numpy()
    
    logger.info("2. Rendering with Monte Carlo...")
    img_mc = generate_mc_drr(volume_tensor, voxel_spacing, res, device, geometry=geo)
    if img_mc is not None:
        images["Monte Carlo"] = img_mc.cpu().squeeze().numpy()
        
    logger.info("3. Rendering with DVR...")
    renderer, cameras = build_dvr_renderer(res, 320, target_size, voxel_spacing, device, geometry=geo)
    if renderer is not None:
        with torch.no_grad():
            img_dvr = renderer(vol, cameras, norm_type="normalized")
            images["DVR"] = img_dvr.cpu().squeeze().numpy()
            
    logger.info("4. Rendering with DiffDRR (Siddon)...")
    drr_siddon, rot, xyz = build_diffdrr_renderer(res, target_size, subject, voxel_spacing, device, geometry=geo)
    if drr_siddon is not None:
        with torch.no_grad():
            img_siddon = drr_siddon(rot, xyz, parameterization="euler_angles", convention="ZXY")
            images["DiffDRR\n(Siddon)"] = img_siddon.cpu().squeeze().numpy()
            
    try:
        from diffdrr.drr import DRR as DiffDRR_module
        logger.info("5. Rendering with DiffDRR (Trilinear)...")
        delx = target_size * voxel_spacing / res
        drr_tri = DiffDRR_module(
            subject,
            sdd=geo.sdd,
            height=res,
            delx=delx,
            renderer="trilinear",
        ).to(device)
        with torch.no_grad():
            img_trilinear = drr_tri(rot, xyz, parameterization="euler_angles", convention="ZXY")
            images["DiffDRR\n(Trilinear)"] = img_trilinear.cpu().squeeze().numpy()
    except Exception as exc:
        logger.debug(f"Skipping DiffDRR Trilinear: {exc}")
        
    logger.info("6. Rendering with DeepDRR...")
    img_deepdrr = generate_deepdrr_drr(volume_tensor, voxel_spacing, res, device, geometry=geo)
    if img_deepdrr is not None:
        images["DeepDRR"] = img_deepdrr.cpu().squeeze().numpy()
        
    results[view_name] = images

In [ ]:
# 3. Plotting results side-by-side
n_views = len(results)
if n_views == 0:
    logger.error("No views were rendered! Cannot create plot.")
else:
    # All views should have roughly the same engines if successful
    all_engines = list(results.values())[0].keys()
    n_engines = len(all_engines)
    
    logger.info(f"Plotting {n_views} views x {n_engines} engines for visual inspection...")
    fig, axes = plt.subplots(n_views, n_engines, figsize=(4 * n_engines, 4 * n_views))
    
    if n_views == 1:
        axes = [axes]
    
    for row_idx, (view_name, images) in enumerate(results.items()):
        for col_idx, engine_name in enumerate(all_engines):
            ax = axes[row_idx][col_idx] if n_engines > 1 else axes[row_idx]
            
            if engine_name in images:
                ax.imshow(images[engine_name], cmap="gray")
            else:
                ax.text(0.5, 0.5, "Failed", ha='center', va='center')
                
            if row_idx == 0:
                ax.set_title(engine_name, fontsize=14, pad=10)
            if col_idx == 0:
                ax.set_ylabel(view_name, fontsize=14, labelpad=10)
                
            ax.set_xticks([])
            ax.set_yticks([])
            
    plt.tight_layout()
    plt.show()